# FinBERT Scoring Input Preparation — Data Collection 

**Academic research only — not investment advice.** This project compares FinBERT-derived news sentiment with RavenPack sentiment and traditional market features for S&P 500 sector ETFs.

This notebook is the Phase 0 handoff between the completed RavenPack extraction and the FinBERT scoring notebook (`07_finbert_sentiment_scoring.ipynb`). It converts the already-extracted and filtered event data into:

1. **`llm_scoring_input.csv`** — one row per distinct headline/text to score with FinBERT exactly once, with a stable `text_id`.
2. **`llm_scoring_event_map.csv`** — one row per event, mapping it to its `text_id` and preserving RavenPack sentiment and timing fields for later aggregation.

No model inference occurs in this notebook. Both outputs are written to `data_collection/raw/`, which is gitignored because it contains licensed WRDS-derived data.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

# Resolve repo-relative paths whether the notebook runs from repo root or data_collection/
CWD = Path.cwd()
REPO_ROOT = CWD if (CWD / "data_collection").exists() else CWD.parent
RAW_DIR = REPO_ROOT / "data_collection" / "raw"

EVENTS_CSV = RAW_DIR / "ravenpack_core_events_2015_2026.csv"
SCORING_INPUT_CSV = RAW_DIR / "llm_scoring_input.csv"
EVENT_MAP_CSV = RAW_DIR / "llm_scoring_event_map.csv"

# Columns that together define a unique piece of text to score.
TEXT_KEYS = ["headline", "event_text"]

print("raw dir:", RAW_DIR)
print("events file exists:", EVENTS_CSV.exists())

raw dir: C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\raw
events file exists: True


## Load the extracted events

Input is the gitignored silver-layer export produced by `01_ravenpack_news_extraction.ipynb`: one row per RavenPack event, with the upstream filters already applied (relevance ≥ 90, event_relevance ≥ 90, rank-1 non-blog sources) and the 4:00 PM ET `signal_calendar_date` already computed for lookahead-safe session mapping.

In [2]:
if not EVENTS_CSV.exists():
    raise FileNotFoundError(
        f"{EVENTS_CSV} not found. This is the gitignored WRDS export produced by "
        "01_ravenpack_news_extraction.ipynb — re-run that notebook first."
    )

events = pd.read_csv(EVENTS_CSV)
print(f"event rows: {len(events):,}")
print("columns:", list(events.columns))
events.head(3)

event rows: 439,743
columns: ['rp_story_id', 'timestamp_utc', 'signal_calendar_date', 'relevance', 'event_relevance', 'rp_source_id', 'source_name', 'topic', 'group_name', 'event_sentiment_score', 'headline', 'event_text', 'source_year']


,rp_story_id,timestamp_utc,signal_calendar_date,relevance,event_relevance,rp_source_id,source_name,topic,group_name,event_sentiment_score,headline,event_text,source_year
0,F54726F720E94F4960B95BF30459D5C3,2020-01-01 00:00:03.144,2020-01-01,100.0,100.0,B5569E,Dow Jones Newswires,economy,balance-of-payments,-0.32,S Korea Dec Exports -5.2% On Year At $45.72B; ...,S Korea Dec Exports -5.2% On Year At $45.72B; ...,2020
1,F7B10F0B144B276261C543BB6BAE0EA8,2020-01-01 00:00:03.151,2020-01-01,100.0,100.0,B5569E,Dow Jones Newswires,economy,balance-of-payments,0.36,S Korea Dec Imports -0.7% On Year At $43.70B; ...,S Korea Dec Imports -0.7% On Year At $43.70B; ...,2020
2,EFAF78E8A157341DB2F18D9A0B0E0EA1,2020-01-01 00:00:03.159,2020-01-01,100.0,100.0,B5569E,Dow Jones Newswires,economy,balance-of-payments,-0.74,S Korea Dec Trade Surplus $2.02B ; Forecast Su...,S Korea Dec Trade Surplus $2.02B ; Forecast Su...,2020


## Normalise text and assign a stable `text_id`

Fill missing `event_text`, strip surrounding whitespace (interior text is left untouched — FinBERT receives the normalized text as-is), then hash `(headline, event_text)` into a deterministic 16-hex-char id. Stability matters: it lets Phase 1 **cache and resume** scoring keyed on `text_id` without ever rescoring a text. A unit-separator byte (`\x1f`) between the two fields ensures `"a"+"bc"` and `"ab"+"c"` don't collide.

In [3]:
def make_text_id(headline: str, event_text: str) -> str:
    """Deterministic 16-hex-char id for a (headline, event_text) pair."""
    payload = f"{headline}\x1f{event_text}".encode("utf-8")
    return hashlib.sha1(payload).hexdigest()[:16]


for col in TEXT_KEYS:
    events[col] = events[col].fillna("").astype(str).str.strip()

events["text_id"] = [
    make_text_id(h, e) for h, e in zip(events["headline"], events["event_text"])
]
events["signal_calendar_date"] = pd.to_datetime(events["signal_calendar_date"])

print(f"distinct text_id: {events['text_id'].nunique():,} "
      f"(from {len(events):,} event rows)")

distinct text_id: 330,518 (from 439,743 event rows)


## Output 1 — distinct texts to score

Collapse to one row per `text_id`. `n_events` (how many event rows share this text) is kept for cost/coverage sizing and is a useful weight later; `first_date`/`last_date` bound where the text appears.

In [4]:
scoring_input = (
    events.groupby("text_id", as_index=False)
    .agg(
        headline=("headline", "first"),
        event_text=("event_text", "first"),
        n_events=("rp_story_id", "size"),
        first_date=("signal_calendar_date", "min"),
        last_date=("signal_calendar_date", "max"),
    )
    .sort_values("n_events", ascending=False)
    .reset_index(drop=True)
)

print(f"distinct texts to score: {len(scoring_input):,}")
scoring_input.head(5)

distinct texts to score: 330,518


,text_id,headline,event_text,n_events,first_date,last_date
0,1e15f4ebe48fa51b,Amundi US Inflation Expectations 10Y UCITS ETF...,Us Inflation Expectations 10Y,1548,2024-01-02,2026-06-30
1,9999669a235a9606,BOJ: Unsecured Overnight Call Rate Data,Bank of Japan announced Wednesday data for the...,639,2020-01-08,2026-06-24
2,a5151d436c47b000,BOJ: Unsecured Overnight Call Rate Data,Bank of Japan announced Thursday data for the ...,635,2020-01-09,2026-06-25
3,4b73410a98f477df,BOJ: Unsecured Overnight Call Rate Data,Bank of Japan announced Friday data for the un...,632,2020-01-10,2026-06-26
4,79ac36b7aa6be844,BOJ: Unsecured Overnight Call Rate Data,Bank of Japan announced Tuesday data for the u...,631,2020-01-07,2026-06-30


## Output 2 — event → `text_id` map

Every event row, reduced to the keys needed to rejoin FinBERT scores and aggregate to sessions later: `rp_story_id`, `timestamp_utc`, `signal_calendar_date`, RavenPack's `event_sentiment_score` (kept as the head-to-head baseline), and `text_id`.

In [5]:
event_map = events[
    [
        "rp_story_id",
        "timestamp_utc",
        "signal_calendar_date",
        "event_sentiment_score",  # RavenPack baseline, kept for comparison
        "text_id",
    ]
].copy()

print(f"event map rows: {len(event_map):,}")
event_map.head(3)

event map rows: 439,743


,rp_story_id,timestamp_utc,signal_calendar_date,event_sentiment_score,text_id
0,F54726F720E94F4960B95BF30459D5C3,2020-01-01 00:00:03.144,2020-01-01,-0.32,0daea5f0e748dd72
1,F7B10F0B144B276261C543BB6BAE0EA8,2020-01-01 00:00:03.151,2020-01-01,0.36,c906f25a686a56bb
2,EFAF78E8A157341DB2F18D9A0B0E0EA1,2020-01-01 00:00:03.159,2020-01-01,-0.74,e814f18fc931efe3


## Integrity checks

Before writing, assert the two outputs reconcile: every event's `text_id` must exist in the scoring input, the per-text event counts must sum back to the total event rows, and `text_id` must be unique in the scoring input.

In [6]:
assert event_map["text_id"].isin(set(scoring_input["text_id"])).all(), \
    "some event text_ids missing from scoring_input"
assert scoring_input["n_events"].sum() == len(event_map), \
    "n_events does not reconcile to event row count"
assert scoring_input["text_id"].is_unique, "text_id not unique in scoring_input"

print("integrity OK — all event text_ids covered, counts reconcile, ids unique")

integrity OK — all event text_ids covered, counts reconcile, ids unique

## Write outputs

Both files land in `data_collection/raw/` (gitignored). Nothing licensed leaves the repo boundary.

In [7]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
scoring_input.to_csv(SCORING_INPUT_CSV, index=False)
event_map.to_csv(EVENT_MAP_CSV, index=False)

n_events, n_texts = len(event_map), len(scoring_input)
print("Phase 0 — LLM scoring input prepared")
print(f"  event rows:              {n_events:,}")
print(f"  distinct texts to score: {n_texts:,}  ({n_texts / n_events:.1%} of rows)")
print(f"  dedup saving:            {n_events - n_texts:,} fewer API calls")
print(f"  date range:              {scoring_input['first_date'].min().date()} "
      f"-> {scoring_input['last_date'].max().date()}")
print(f"  wrote {SCORING_INPUT_CSV}")
print(f"  wrote {EVENT_MAP_CSV}")

Phase 0 — LLM scoring input prepared
  event rows:              439,743
  distinct texts to score: 330,518  (75.2% of rows)
  dedup saving:            109,225 fewer API calls
  date range:              2020-01-01 -> 2026-07-01
  wrote C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\raw\llm_scoring_input.csv
  wrote C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\raw\llm_scoring_event_map.csv


## Data-quality note for Phase 1

The most frequent "headlines" in this macro feed are **boilerplate data-feed prints** that carry no narrative sentiment — e.g. *"BOJ: Unsecured Overnight Call Rate Data"* and ETF *"Net Asset Value(s)"* notices. They are noise for both RavenPack and an LLM, and are part of why the current baseline is a null result.

Consider filtering these before Phase 1 to save API cost, **or** score them and flag them as a data-quality finding in the report. Inspect the highest-frequency texts below to decide.

In [8]:
scoring_input[["n_events", "headline", "event_text"]].head(20)

,n_events,headline,event_text
0,1548,Amundi US Inflation Expectations 10Y UCITS ETF...,Us Inflation Expectations 10Y
1,639,BOJ: Unsecured Overnight Call Rate Data,Bank of Japan announced Wednesday data for the...
2,635,BOJ: Unsecured Overnight Call Rate Data,Bank of Japan announced Thursday data for the ...
3,632,BOJ: Unsecured Overnight Call Rate Data,Bank of Japan announced Friday data for the un...
4,631,BOJ: Unsecured Overnight Call Rate Data,Bank of Japan announced Tuesday data for the u...
5,583,BOJ: Unsecured Overnight Call Rate Data,Bank of Japan announced Monday data for the un...
6,529,Amundi US Inflation Expectations 10Y UCITS ETF...,Us Inflation Expectations 10Y
7,528,Press Release: Fitch Takes Various Rating Acti...,Fitch Takes Various Rating Actions on U.S.
8,520,Fitch Takes Various Rating Actions on U.S. Enh...,Fitch Takes Various Rating Actions on U.S.
9,323,USDA U.S. Livestock Imports from Canada -2-,U.S. Livestock Imports from Canada


---

**Next — Phase 1 (`07_llm_sentiment_scoring.ipynb`):** load `llm_scoring_input.csv`, score each distinct text once with FinBERT (JSON out: `sentiment` −1..1, `confidence`, `rationale`; low temperature; batched; cache/resume on `text_id`), then join scores back through `llm_scoring_event_map.csv` and aggregate to the trading-session grain using the same 4 PM ET cutoff rules as the existing pipeline.